# ConocoPhillips ML Research, Rebuilt

This notebook is the cleaned-up sequel to the original Colab notebook. The original work had good instincts, but it mixed targets, used non-reproducible Drive paths, and compared models on inconsistent setups. This version turns the project into a repeatable ERCOT forecasting workflow driven by official EIA bulk data.

**What changed**

- Replaced hard-coded Google Drive inputs with a local cache built from official EIA bulk files.
- Standardized the task as a **24-hour-ahead hourly ERCOT load forecast**.
- Split the project into train, validation, and test windows with an extra rolling backtest.
- Benchmarked naive, published day-ahead, linear, and boosted models on the same target.
- Focused the project on an operational use case: **improving the published day-ahead load forecast instead of forecasting blindly**.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
hourly_df = pd.read_csv(ROOT / "data" / "ercot_eia_hourly.csv")
hourly_df["timestamp_utc"] = pd.to_datetime(hourly_df["timestamp_utc"], utc=True, format="ISO8601")
metrics_df = pd.read_csv(ROOT / "artifacts" / "forecast_metrics.csv")
backtest_df = pd.read_csv(ROOT / "artifacts" / "rolling_backtest.csv")
hourly_df.head()

## Data Scope

- Official EIA ERCOT demand series used: `EBA.ERCO-ALL.D.H`
- Official EIA ERCOT day-ahead forecast used: `EBA.ERCO-ALL.DF.H`
- Coverage: **2015-07-01T06:00:00+00:00** through **2026-03-10T18:00:00+00:00**
- Total hourly observations after alignment: **93,588**

The new notebook does not try to perfectly recreate the original Colab data stack because those Google Drive files were not present in the workspace. Instead, it rebuilds the project around the most reliable public ERCOT load signal I could recover end-to-end.

## Forecast Definition

The key reframing is simple: we forecast each hour using only information that should already be available 24 hours earlier.

- Calendar features come from the target hour.
- Lag features come from actual ERCOT load 24 hours or more in the past.
- The published day-ahead load forecast is treated as an exogenous operational prior.

That makes the exercise much more realistic than asking a blind model to extrapolate the whole future horizon with no live system context.

In [ ]:
feature_example = hourly_df.copy()
feature_example["lag_24"] = feature_example["actual"].shift(24)
feature_example["lag_168"] = feature_example["actual"].shift(168)
feature_example["forecast_gap_24"] = feature_example["day_ahead"] - feature_example["lag_24"]
feature_example[["timestamp_utc", "actual", "day_ahead", "lag_24", "lag_168", "forecast_gap_24"]].head()

## Validation Results

| model | rmse | mae | wape | note |
| --- | --- | --- | --- | --- |
| blend_ridge_direct_hgb_direct | 1630.76 | 1200.47 | 2.21 | ridge_direct=0.70 |
| ridge_direct | 1642.89 | 1218.97 | 2.25 | alpha=20.0 |
| hgb_direct | 1707.02 | 1247.96 | 2.30 | learning_rate=0.03,max_depth=12,max_iter=450,min_samples_leaf=30,l2_regularization=0.2 |
| hgb_residual_on_day_ahead | 1735.51 | 1276.50 | 2.35 | learning_rate=0.03,max_depth=10,max_iter=350,min_samples_leaf=25,l2_regularization=0.1 |
| day_ahead | 1807.25 | 1299.95 | 2.39 |  |
| naive_24 | 3017.00 | 2211.34 | 4.07 |  |
| avg_24_168 | 3298.36 | 2604.59 | 4.80 |  |
| naive_168 | 5081.26 | 4000.61 | 7.37 |  |

## Test Results

| model | rmse | mae | wape | note |
| --- | --- | --- | --- | --- |
| blend_ridge_direct_hgb_direct | 1362.08 | 953.24 | 1.86 | ridge_direct=0.70 |
| ridge_direct | 1397.39 | 952.41 | 1.86 | alpha=20.0 |
| hgb_direct | 1476.59 | 1073.64 | 2.10 | learning_rate=0.03,max_depth=12,max_iter=450,min_samples_leaf=30,l2_regularization=0.2 |
| hgb_residual_on_day_ahead | 1497.40 | 1101.82 | 2.15 | learning_rate=0.03,max_depth=10,max_iter=350,min_samples_leaf=25,l2_regularization=0.1 |
| day_ahead | 1634.62 | 1097.10 | 2.15 |  |
| naive_24 | 3560.56 | 2530.78 | 4.95 |  |
| avg_24_168 | 4098.38 | 2910.96 | 5.69 |  |
| naive_168 | 6375.20 | 4591.57 | 8.98 |  |

## Headline Takeaways

- Best holdout model: **blend_ridge_direct_hgb_direct**
- RMSE improvement vs published day-ahead forecast: **16.67%**
- RMSE improvement vs naive 24-hour lag: **61.75%**

The biggest conceptual win is that the project is no longer asking “which generic time-series model looks smartest?” It is asking “how do we systematically improve a load forecast that the market already publishes?” That is a much more actionable framing.

![Recent Forecast Window](artifacts/recent_forecast_window.png)

![Model RMSE](artifacts/model_test_rmse.png)

## Rolling Backtest

| fold | day_ahead | naive_24 | ridge_direct |
| --- | --- | --- | --- |
| fold_1 | 1355.58 | 3112.90 | 1268.07 |
| fold_2 | 1266.23 | 2778.36 | 1214.90 |
| fold_3 | 2732.43 | 5072.10 | 2217.64 |
| fold_4 | 1757.05 | 4454.47 | 1321.54 |
| fold_5 | 854.43 | 1932.76 | 780.85 |
| fold_6 | 1127.51 | 2949.96 | 1054.37 |

![Rolling Backtest](artifacts/rolling_backtest_rmse.png)

## Error Anatomy

                The best model wins by correcting structured bias in the published day-ahead forecast, especially around specific local hours.

                - Hour 11: 263.3 lower MAE than the day-ahead baseline.
- Hour 09: 261.1 lower MAE than the day-ahead baseline.
- Hour 10: 253.4 lower MAE than the day-ahead baseline.

                ![Hourly Error Profile](artifacts/hourly_mae_profile.png)

## Feature Importance

| feature | importance |
| --- | --- |
| lag_24 | 3536.0321 |
| day_ahead | 3520.6501 |
| lag_168 | 3288.1186 |
| forecast_gap_168 | 530.7528 |
| forecast_gap_24 | 399.7136 |
| roll_mean_336 | 252.6722 |
| lag_spread_24_168 | 198.4319 |
| roll_mean_48 | 125.7111 |
| lag_48 | 117.8938 |
| roll_mean_24 | 101.7993 |
| roll_std_24 | 90.0201 |
| roll_std_336 | 30.9428 |

![Feature Importance](artifacts/feature_importance.png)

## Legacy Notebook Snapshot

| legacy_model | rmse | note |
| --- | --- | --- |
| ARIMA | 15712.27 | Original notebook output; different data slice and target. |
| SARIMA | 3977651.41 | Original notebook used cumulative monthly demand. |
| Prophet | 5332.77 | Best saved result inside the original notebook. |
| Prophet (bivariate) | 6418.70 | Original notebook output. |
| VAR | 6884.17 | Original notebook output. |

These legacy scores are helpful context, but they are not directly comparable because the old notebook mixed different targets, frequencies, and data sources.

## Novel Insights

- The published ERCOT day-ahead forecast is already a stronger baseline than a pure historical lag model, which means the highest-value ML move is **forecast correction**, not blind forecasting.
- Weekly seasonality matters, but the 24-hour lag is much stronger than the 168-hour lag on this test window, so short-term autocorrelation is still carrying most of the signal.
- The gains are not uniform across the day. The model earns its keep in the hours where the system ramps and where market forecasts tend to accumulate directional bias.
- A regularized linear model remains highly competitive here because the engineered features and the operational forecast prior do most of the heavy lifting.

## Next Pushes

- Bring in weather forecasts, not just realized weather, so the project stays operationally fair.
- Move from a single 24-hour-ahead target to a direct multi-horizon setup.
- Add probabilistic intervals so the notebook can speak to risk, not just point accuracy.
- If the original proprietary ERCOT + weather spreadsheets reappear, rerun this framework on them so we can make apples-to-apples comparisons with the first Colab notebook.